# Trading de energía con aprendizaje automático, sobre el mercado español real

Este cuaderno se escribió primero con datos simulados y daba dos resultados
limpios. Una revisión adversarial encontró que los dos venían del simulador y
no del mundo: su estacionalidad era un coseno determinista idéntico cada año, y
la habilidad de sus previsiones meteorológicas era un parámetro escrito a mano.

Está rehecho con **OMIE y MIBGAS reales**: ocho años y medio de casaciones del
pool ibérico y del gas español, más la temperatura de Madrid. Lo que sale es
menos vendible y mucho más interesante.

El resultado central no es que haya menos señal de la que parecía. Es que la
señal más grande del mercado eléctrico español es enorme, perfectamente
medible, **y no se puede cobrar**. Entender por qué es el objeto de las
secciones 3 y 4.

## Lo que hay y lo que no

| Disponible | Falta, y no se sustituye |
|---|---|
| OMIE, 102 143 casaciones desde 2018 | Generación eólica y demanda (ESIOS pide token) |
| MIBGAS, 3 877 días desde 2016 | Previsiones meteorológicas publicadas |
| Temperatura de Madrid, 3 078 días | Precio del CO2 (EUA) y TTF |

Esa segunda columna importa: **sin previsiones publicadas, la afirmación «la
señal está en las previsiones y no en el historial» no se puede contrastar**.
Se contrasta su otra mitad, que es la que la sostenía.

In [1]:
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 110)

from icm.datos.ficheros import cargar_precios_fichero
from icm.energia.simulacion import grados_dia

RUTA = "../datos/reales"
DIAS = ["lunes", "martes", "miercoles", "jueves", "viernes", "sabado", "domingo"]

pool = cargar_precios_fichero(f"{RUTA}/omie_diario.csv", ["ES"])["ES"].rename("pool")
gas = cargar_precios_fichero(f"{RUTA}/mibgas_diario.csv", ["ES"])["ES"].rename("gas")
temperatura = pd.read_csv(
    f"{RUTA}/meteo_madrid.csv", parse_dates=["fecha"]
).set_index("fecha")["temperatura"]

print(f"pool  {len(pool):>6,} dias  {pool.index.min():%Y-%m-%d} .. {pool.index.max():%Y-%m-%d}")
print(f"gas   {len(gas):>6,} dias  {gas.index.min():%Y-%m-%d} .. {gas.index.max():%Y-%m-%d}")
print(f"temp  {len(temperatura):>6,} dias")

pool   3,185 dias  2018-01-01 .. 2026-09-25
gas    3,877 dias  2016-01-06 .. 2026-09-26
temp   3,078 dias


## 1. El mercado, sin adornos

Tres cosas que ningún simulador de este proyecto reproducía.

In [2]:
print("Nivel del pool (EUR/MWh) por anio:")
resumen_anual = pd.DataFrame({
    "media": pool.groupby(pool.index.year).mean(),
    "maximo": pool.groupby(pool.index.year).max(),
    "minimo": pool.groupby(pool.index.year).min(),
    "vol_anual": np.log(pool).diff().groupby(pool.index.year).std() * np.sqrt(365),
})
print(resumen_anual.round(2).to_string())

print()
print(f"Pico historico: {pool.max():.2f} EUR/MWh el {pool.idxmax():%d-%m-%Y}")
print("  (es el dia siguiente al inicio de la invasion de Ucrania)")
print(f"Minimo diario:  {pool.min():.4f} EUR/MWh el {pool.idxmin():%d-%m-%Y}")

Nivel del pool (EUR/MWh) por anio:
        media  maximo  minimo  vol_anual
fecha                                   
2018    57.29   75.93    4.50       4.10
2019    47.68   69.43    1.94       4.06
2020    33.96   56.87    4.63       4.27
2021   111.93  383.67    1.42       6.97
2022   167.53  544.98    2.65       4.61
2023    87.20  151.43    1.51       9.32
2024    63.03  146.67    0.44      12.11
2025    65.16  144.92    1.72       9.60
2026    72.64  177.44    0.18       9.78

Pico historico: 544.98 EUR/MWh el 08-03-2022
  (es el dia siguiente al inicio de la invasion de Ucrania)
Minimo diario:  0.1778 EUR/MWh el 29-03-2026


La volatilidad anualizada del rendimiento logarítmico diario del pool ronda
**el 600 %**. No es un error de cálculo ni un artefacto: un mercado eléctrico
mayorista no es un activo financiero, es el precio de un bien que no se puede
almacenar y cuyo precio se forma de nuevo cada día. Volveremos a esto.

In [3]:
intradiario = cargar_precios_fichero(
    f"{RUTA}/omie_intradiario.csv", ["ES"], columna_hora="periodo", huso="Europe/Madrid"
)["ES"]

no_positivos = intradiario <= 0
print(f"Casaciones a cero o negativas: {int(no_positivos.sum()):,} de {len(intradiario):,} "
      f"({100 * no_positivos.mean():.2f} %)")
print(f"Minimo: {intradiario.min():.2f} EUR/MWh")
print()
print("Cuota de casaciones no positivas por anio:")
cuota = no_positivos.groupby(intradiario.index.year).mean() * 100
print(cuota.round(2).to_string())

Casaciones a cero o negativas: 5,868 de 102,143 (5.74 %)
Minimo: -15.00 EUR/MWh

Cuota de casaciones no positivas por anio:
fecha
2018     0.00
2019     0.00
2020     0.00
2021     0.00
2022     0.03
2023     1.18
2024     8.93
2025     6.18
2026    15.76


Ahí está la firma del crecimiento renovable: de cero antes de 2022 a más del
10 % de las casaciones. Y es la razón por la que
`rendimientos(precios, tipo="log")` se niega a trabajar con la serie
intradiaria: el logaritmo de cero no existe. La serie **diaria** sí admite el
logaritmo, porque la media aritmética del día nunca baja de 0,18 €/MWh.

El tercer rasgo es de calendario: el 1 de octubre de 2025 OMIE pasó de casar 24
períodos horarios al día a casar 96 cuarto-horarios.

In [4]:
periodos_por_dia = intradiario.groupby(intradiario.index.normalize()).size()
print("Dias por numero de periodos de casacion:")
print(periodos_por_dia.value_counts().sort_index().to_string())
print()
print("(24 y 96 son los dias normales de cada regimen; 23/25 y 92/100, los del cambio de hora)")

Dias por numero de periodos de casacion:
ES
23        8
24     2813
25        7
92        1
96      355
100       1

(24 y 96 son los dias normales de cada regimen; 23/25 y 92/100, los del cambio de hora)


## 2. El spark spread español, por primera vez con datos reales

MIBGAS y OMIE están los dos en €/MWh y comparten ocho años y medio. Es el
cálculo que el cuaderno anterior hacía con precios inventados.

In [5]:
from icm.energia.spreads import PLANTAS, horas_en_dinero, spark_spread, valor_opcionalidad

planta = PLANTAS["ciclo_combinado"]
print(planta)

comun = pool.index.intersection(gas.index)
spread = spark_spread(pool.loc[comun], gas.loc[comun], planta.eficiencia,
                      costes_variables=planta.costes_variables)

print()
print(f"Muestra comun: {len(spread):,} dias")
print(f"  media    {spread.mean():+8.2f} EUR/MWh_e")
print(f"  mediana  {spread.median():+8.2f}")
print(f"  desv.    {spread.std():8.2f}")
print(f"  minimo   {spread.min():+8.2f}  el {spread.idxmin():%d-%m-%Y}")
print(f"  maximo   {spread.max():+8.2f}  el {spread.idxmax():%d-%m-%Y}")
print(f"  asimetria {spread.skew():+7.2f}   curtosis en exceso {spread.kurtosis():+.2f}")
print(f"  dias en dinero: {horas_en_dinero(spread):.1%}")

ParametrosPlanta(nombre='Ciclo combinado de gas (referencia de mercado, 50 %)', eficiencia=0.5, factor_emision=0.2, costes_variables=2.0)

Muestra comun: 3,153 dias
  media       -1.85 EUR/MWh_e
  mediana    +10.32
  desv.       39.25
  minimo    -308.86  el 28-08-2022
  maximo    +114.26  el 08-03-2022
  asimetria   -2.12   curtosis en exceso +7.37
  dias en dinero: 66.3%


La media es **negativa** y la mediana claramente positiva. Eso no es una
contradicción: es una distribución con cola izquierda gruesa (asimetría −2,1),
y es exactamente la forma que hace valiosa la opción de no arrancar.

Con el simulador la media era +23,3 €/MWh y el 90 % de los días estaba en
dinero. El ciclo combinado español real **vive sobre la línea de equilibrio**,
no holgadamente por encima.

In [6]:
optatividad = float(valor_opcionalidad(spread).mean())
print(f"Valor de la opcionalidad E[max(S,0)]: {optatividad:+.2f} EUR/MWh_e")
print(f"Margen de operar siempre  max(E[S],0): {max(spread.mean(), 0.0):+.2f}")
print(f"Brecha de Jensen: {optatividad - max(spread.mean(), 0.0):+.2f}")
print()
print("Con el simulador la brecha era de 1,21 EUR/MWh_e. Aqui es diez veces mayor,")
print("y toda la diferencia esta en la cola izquierda que el coseno no genera.")

Valor de la opcionalidad E[max(S,0)]: +12.74 EUR/MWh_e
Margen de operar siempre  max(E[S],0): +0.00
Brecha de Jensen: +12.74

Con el simulador la brecha era de 1,21 EUR/MWh_e. Aqui es diez veces mayor,
y toda la diferencia esta en la cola izquierda que el coseno no genera.


### El detalle que cambia el signo

Hay 563 días de la muestra —el 17,9 %— en los que este cálculo **no mide el
margen de la planta**: del 15 de junio de 2022 al 31 de diciembre de 2023
estuvo en vigor el mecanismo de ajuste ibérico, con el gas topado en la
formación del precio del pool y la compensación a las centrales pagada *fuera*
del pool.

In [7]:
tope = (spread.index >= "2022-06-15") & (spread.index <= "2023-12-31")
print(f"Dias bajo el mecanismo de ajuste iberico: {int(tope.sum()):,} ({100 * tope.mean():.1f} %)")
print()
for etiqueta, mascara in (("muestra completa", slice(None)), ("dentro del tope", tope), ("fuera del tope", ~tope)):
    s = spread[mascara] if not isinstance(mascara, slice) else spread
    print(f"  {etiqueta:18s} media {s.mean():+7.2f}   mediana {s.median():+7.2f}   "
          f"en dinero {horas_en_dinero(s):5.1%}   n={len(s):,}")

Dias bajo el mecanismo de ajuste iberico: 563 (17.9 %)

  muestra completa   media   -1.85   mediana  +10.32   en dinero 66.3%   n=3,153
  dentro del tope    media  -25.85   mediana   -2.78   en dinero 48.7%   n=563
  fuera del tope     media   +3.37   mediana  +10.82   en dinero 70.2%   n=2,590


Quitando esa ventana, el spread medio **cambia de signo**. No es un detalle de
limpieza de datos: es que durante año y medio el precio del pool español dejó
de ser el ingreso de un ciclo combinado, y ningún análisis que no lo sepa
puede interpretar esa media.

Es el tipo de cosa que solo aparece con datos reales, y la razón por la que el
conocimiento del mercado no es opcional.

In [8]:
from icm.energia.spreads import spark_spread_limpio

print("Sensibilidad al precio del CO2 (no tenemos la serie del EUA, asi que se barre):")
print()
print(f"{'EUA (EUR/t)':>12}  {'media':>8}  {'mediana':>8}  {'en dinero':>10}")
for precio_co2 in (0.0, 20.0, 50.0, 80.0, 100.0):
    limpio = spark_spread_limpio(
        pool.loc[comun], gas.loc[comun], pd.Series(precio_co2, index=comun),
        planta.eficiencia, planta.factor_emision,
        costes_variables=planta.costes_variables,
    )
    print(f"{precio_co2:12.0f}  {limpio.mean():+8.2f}  {limpio.median():+8.2f}  {horas_en_dinero(limpio):10.1%}")
print()
print(f"Factor de emision de la planta: {planta.factor_emision} tCO2 por MWh termico")
print(f"  que al {planta.eficiencia:.0%} de rendimiento son "
      f"{planta.factor_emision / planta.eficiencia:.2f} tCO2 por MWh electrico")

Sensibilidad al precio del CO2 (no tenemos la serie del EUA, asi que se barre):

 EUA (EUR/t)     media   mediana   en dinero
           0     -1.85    +10.32       66.3%
          20     -9.85     +2.32       55.1%
          50    -21.85     -9.68       25.7%
          80    -33.85    -21.68       10.5%
         100    -41.85    -29.68        4.5%

Factor de emision de la planta: 0.2 tCO2 por MWh termico
  que al 50% de rendimiento son 0.40 tCO2 por MWh electrico


## 3. ¿Hay algo que se pueda predecir? Sí, y es enorme

La pregunta del cuaderno original. Con datos reales la respuesta es que sí, y
la señal es mucho mayor de lo que cualquier simulador de este proyecto
contenía. Empecemos por lo que el mercado hace de forma más obvia.

In [9]:
rend = np.log(pool).diff().dropna().rename("rendimiento")
por_dia = pd.DataFrame({
    "nivel_medio": pool.groupby(pool.index.dayofweek).mean(),
    "rend_medio": rend.groupby(rend.index.dayofweek).mean(),
    "t": rend.groupby(rend.index.dayofweek).apply(
        lambda s: s.mean() / (s.std(ddof=1) / np.sqrt(len(s)))
    ),
    "n": rend.groupby(rend.index.dayofweek).size(),
})
por_dia.index = DIAS
print("El ciclo semanal del pool iberico:")
print(por_dia.round(4).to_string())

El ciclo semanal del pool iberico:
           nivel_medio  rend_medio        t    n
lunes          81.8395      0.3193  13.3517  455
martes         84.5027      0.0463   3.1363  456
miercoles      82.8569     -0.0130  -0.8269  456
jueves         83.0538     -0.0159  -1.2140  454
viernes        81.1998     -0.0429  -2.7808  456
sabado         71.7070     -0.1724  -9.2077  455
domingo        65.3198     -0.1155  -5.9946  452


El lunes sube un **+32 %** logarítmico de media con un estadístico t de 13,4, y
el sábado baja un −17 % con t de −9,2. Son efectos enormes y estadísticamente
aplastantes. El nivel lo dice más claro: el domingo el pool vale 65 €/MWh y el
martes 84,50. Diecinueve euros de recorrido sobre un nivel medio de 79.

Esto no es un descubrimiento: es la demanda. Las fábricas y las oficinas no
consumen el domingo. Cualquiera del sector lo sabe, y el simulador de este
proyecto **no podía contenerlo**, porque genera solo días hábiles.

Lo interesante es lo que pasa cuando se lo damos a un modelo.

In [10]:
from icm.ml.caracteristicas import calendario

X_cal = calendario(rend.index)
print(f"Solo calendario: {X_cal.shape[1]} columnas -> {list(X_cal.columns)}")

Solo calendario: 8 columnas -> ['dia_martes', 'dia_miercoles', 'dia_jueves', 'dia_viernes', 'dia_sabado', 'dia_domingo', 'sin_anio', 'cos_anio']


In [11]:
from icm.ml.estrategia import ejecutar_estrategia, resumen_estrategia
from icm.ml.evaluacion import error_escalado_medio
from icm.ml.modelos import crear_modelo
from icm.ml.validacion import ParticionWalkForward, validacion_walk_forward

objetivo = rend.shift(-1).dropna().rename("rend_futuro_1")
objetivo.attrs["horizonte"] = 1
indice = X_cal.index.intersection(objetivo.index)
particion = ParticionWalkForward(n_particiones=5, embargo=1)

rezagos = pd.DataFrame({f"rend_ret{k}": rend.shift(k) for k in (1, 2, 3, 5)}).dropna()
indice = X_cal.index.intersection(rezagos.index).intersection(objetivo.index)

def evaluar(X, nombre):
    resultado = validacion_walk_forward(
        crear_modelo("ridge"), X.loc[indice], objetivo.loc[indice], particion
    )
    pred = resultado.predicciones.dropna()
    real = objetivo.loc[pred.index]
    fila = resumen_estrategia(ejecutar_estrategia(pred, real, coste_bps=2.0))
    return {
        "configuracion": nombre,
        "MASE": error_escalado_medio(real, pred),
        "Sharpe": fila["ratio_sharpe"],
        "rent. anual": fila["rentabilidad_anualizada"],
        "vol. anual": fila["volatilidad_anualizada"],
        "maxima caida": fila["maxima_caida"],
    }

filas = [
    evaluar(X_cal, "A. solo calendario"),
    evaluar(rezagos, "B. solo rezagos del precio"),
    evaluar(X_cal.join(rezagos, how="inner"), "C. calendario + rezagos"),
]
print(pd.DataFrame(filas).round(4).to_string(index=False))

             configuracion   MASE  Sharpe  rent. anual  vol. anual  maxima caida
        A. solo calendario 0.6560  0.1554      -1.0000      6.4018       -6.0523
B. solo rezagos del precio 0.6805  1.3791      -0.9986      6.3784      -24.3089
   C. calendario + rezagos 0.6495  0.9909      -1.0000      6.3898       -1.0333


Fijaos en las columnas que el cuaderno anterior no informaba.

El Sharpe de las tres configuraciones va de 0,16 a 1,38, pero la
**rentabilidad anualizada es −1,000** en las tres, y las caídas máximas son de
−6, −24 y −1. Una rentabilidad anualizada de −1 significa que el capital
compuesto llega a **cero o por debajo**: la estrategia no pierde, aniquila.

Eso pasa porque la volatilidad anualizada del rendimiento diario del pool es
de **640 %**. Una posición de una unidad sobre una serie así no es una
estrategia, es la ruina: basta un día en que el precio pase de 0,18 a 80 €/MWh
(un rendimiento logarítmico de +6) para que el lado corto pierda seis veces el
capital.

**Y el ratio de Sharpe no ve nada de eso.** Es el cociente entre una media
aritmética y una desviación típica, y es ciego a la ruina. El cuaderno anterior
informaba el Sharpe y no estas dos columnas.

Antes de arreglarlo, una nota sobre el aparato de aprendizaje automático.

In [12]:
# Una regla de CERO parametros: los signos los pone el conocimiento del sector.
# Lunes arriba, fin de semana abajo. No se estima nada.
posicion = pd.Series(0.0, index=objetivo.index)
posicion[objetivo.index.dayofweek == 6] = +1.0   # decision del domingo -> entrega del lunes
posicion[objetivo.index.dayofweek == 4] = -1.0   # viernes -> sabado
posicion[objetivo.index.dayofweek == 5] = -1.0   # sabado -> domingo

regla = resumen_estrategia(ejecutar_estrategia(posicion, objetivo, coste_bps=2.0, retardo=0))
print(f"Regla de cero parametros:  Sharpe {regla['ratio_sharpe']:+.3f}   "
      f"rent. anual {regla['rentabilidad_anualizada']:+.3f}   "
      f"maxima caida {regla['maxima_caida']:+.3g}")
print(f"Ridge con calendario:      Sharpe {filas[0]['Sharpe']:+.3f}   "
      f"rent. anual {filas[0]['rent. anual']:+.3f}   "
      f"maxima caida {filas[0]['maxima caida']:+.3g}")
print()
print("Tres signos puestos a mano por conocimiento del sector dan un Sharpe")
print("28 veces mayor que toda la maquinaria de validacion y regularizacion.")
print("Y una caida maxima que ya no es un numero, sino un desbordamiento:")
print("el capital compuesto cruza el cero y la caida relativa deja de existir.")

Regla de cero parametros:  Sharpe +4.412   rent. anual -1.000   maxima caida -2.52e+36
Ridge con calendario:      Sharpe +0.155   rent. anual -1.000   maxima caida -6.05

Tres signos puestos a mano por conocimiento del sector dan un Sharpe
28 veces mayor que toda la maquinaria de validacion y regularizacion.
Y una caida maxima que ya no es un numero, sino un desbordamiento:
el capital compuesto cruza el cero y la caida relativa deja de existir.


## 4. Dimensionar la posición, y la pregunta que queda

El problema de la sección anterior no es la señal: es el tamaño de la apuesta.
Con la volatilidad objetivo que ya usaba el cuaderno anterior —estimando la
volatilidad **solo con el pasado**, que fue un error que hubo que corregir dos
veces— la misma regla deja de ser ruinosa.

In [13]:
# La volatilidad se estima con una ventana que termina AYER: usar la del propio
# dia seria mirar al futuro, y es el fallo que este cuaderno cometio dos veces.
vol_pasada = rend.rolling(60).std().shift(1) * np.sqrt(252)
escala = (0.10 / vol_pasada).clip(upper=1.0)
dimensionada = (posicion * escala.reindex(posicion.index)).dropna()

dim = resumen_estrategia(ejecutar_estrategia(
    dimensionada, objetivo.loc[dimensionada.index], coste_bps=2.0, retardo=0,
    modo="proporcional", escala=1.0, limite=10.0,
))
print("La misma regla, con objetivo de volatilidad del 10 % anual:")
print(f"  Sharpe            {dim['ratio_sharpe']:+.3f}")
print(f"  rent. anualizada  {dim['rentabilidad_anualizada']:+.3f}")
print(f"  vol. realizada    {dim['volatilidad_anualizada']:.3f}")
print(f"  maxima caida      {dim['maxima_caida']:+.4f}")

La misma regla, con objetivo de volatilidad del 10 % anual:
  Sharpe            +4.870
  rent. anualizada  +0.640
  vol. realizada    0.103
  maxima caida      -0.0468


Ahora sí: volatilidad realizada del 10,3 % frente al 10 % pedido, rentabilidad
compuesta de **+64 % anual** y caída máxima del 4,7 %. Sobre el papel, es una
estrategia excelente.

Queda una sola pregunta, y no es estadística.

### ¿Se puede tener esta posición?

Mirad primero qué pasa al sumar las siete medias semanales.

In [14]:
medias = rend.groupby(rend.index.dayofweek).mean()
suma = medias.sum()
recorrido = medias.abs().sum()
print(f"Suma de las siete medias semanales: {suma:+.6f}")
print(f"Suma de sus valores absolutos:      {recorrido:.4f}")
print(f"La suma neta es el {100 * abs(suma) / recorrido:.2f} % del recorrido")

Suma de las siete medias semanales: +0.005895
Suma de sus valores absolutos:      0.7253
La suma neta es el 0.81 % del recorrido


**El patrón es un bucle cerrado.** El lunes sube lo que el fin de semana baja,
con una deriva neta del 0,8 % del recorrido. No hay tendencia que cobrar: hay
forma. Y para cobrar una forma hay que estar dentro y salir.

> Para cobrar el +32 % del domingo al lunes hay que **comprar energía con
> entrega el domingo y venderla con entrega el lunes**.

Eso no es una posición. Son dos entregas distintas de un bien que no se puede
almacenar. El «rendimiento» de un día al siguiente en el mercado diario no es
el rendimiento de nada que se pueda tener: **no existe el instrumento que lo
pague**.

Cuando `ejecutar_estrategia` calcula `posicion[t] * rendimiento[t]` sobre el
precio *spot* eléctrico, está tratando la electricidad como si fuera una
acción. Con una acción, mantenerla de un día para otro es gratis y el
rendimiento es tuyo. Con la electricidad, mantenerla es físicamente imposible.

Así que ese +64 % anual mide **la amplitud de una forma de calendario**, no una
rentabilidad. Es el mismo tipo de error que la revisión encontró en el
simulador —un número con aspecto de dinero que no lo es— por una razón
distinta: allí era una previsión fabricada, aquí es un subyacente que no se
puede almacenar.

### La comprobación: el gas sí se almacena

Si el argumento es correcto, la misma regla aplicada a un bien **almacenable**
debería dar un resultado cobrable. El gas se almacena — es el negocio de los
almacenamientos subterráneos — y MIBGAS es además un producto de liquidación
financiera.

In [15]:
def regla_semanal(serie, etiqueta):
    r = np.log(serie).diff().dropna()
    y = r.shift(-1).dropna()
    y.attrs["horizonte"] = 1
    pos = pd.Series(0.0, index=y.index)
    pos[y.index.dayofweek == 6] = +1.0
    pos[y.index.dayofweek.isin([4, 5])] = -1.0
    crudo = resumen_estrategia(ejecutar_estrategia(pos, y, coste_bps=2.0, retardo=0))
    return {
        "serie": etiqueta,
        "almacenable": "no" if etiqueta == "OMIE (pool)" else "si",
        "Sharpe": crudo["ratio_sharpe"],
        "rent. anual": crudo["rentabilidad_anualizada"],
        "maxima caida": crudo["maxima_caida"],
    }

comparacion = pd.DataFrame([regla_semanal(pool, "OMIE (pool)"), regla_semanal(gas, "MIBGAS (gas)")])
print(comparacion.to_string(index=False))

       serie almacenable   Sharpe  rent. anual  maxima caida
 OMIE (pool)          no 4.411614    -1.000000 -2.521287e+36
MIBGAS (gas)          si 2.550177     3.131199 -5.723460e-01


La misma regla, sin dimensionar, sobre los dos mercados:

- En el **pool**, Sharpe +4,4 con rentabilidad compuesta de −100 % y una caída
  que desborda. Ruina.
- En el **gas**, Sharpe +2,6 con rentabilidad compuesta de **+313 % anual** y
  caída máxima del 57 %. Duro, pero sobrevive.

El Sharpe es *mejor* en el mercado donde no se puede cobrar nada y *peor* en el
que sí. Esa inversión es la mejor prueba de que el ratio de Sharpe no es la
medida adecuada aquí, y de que la pregunta «¿se puede almacenar el subyacente?»
decide más que cualquier estadístico.

### Qué sí se operaría en el pool

Productos con entrega: futuros mensuales y trimestrales, la planta de ciclo
combinado de la sección 2 —que es una opción sobre el spread—, o una batería y
un bombeo, que es literalmente comprar barato el domingo para vender caro el
lunes. Por esa vía el ciclo semanal **sí** se cobra, y su límite es la
capacidad de almacenamiento y su eficiencia de ciclo, no la calidad de la
predicción.

Para eso no hace falta este modelo. Hace falta un activo físico.

## 5. La tesis del cuaderno anterior, contrastada

El cuaderno afirmaba que las variables exógenas actúan de forma
**contemporánea**, que sus valores rezagados no predicen, y que la señal útil
está en las **previsiones publicadas**. La revisión demostró que la tercera
pata era circular por construcción: el precio simulado se fijaba sobre el
valor realizado de la exógena, y la previsión se fabricaba a partir de ese
mismo valor futuro.

Con datos reales no hay archivo de previsiones publicadas, así que **la tercera
pata no se puede medir**. Las dos primeras sí.

In [16]:
gd = grados_dia(temperatura)
hdd, cdd = gd["hdd"], gd["cdd"]

def correlaciones(serie_precio, exogena, nombre):
    r = np.log(serie_precio).diff().dropna()
    cambio = exogena.diff()
    idx = r.index.intersection(cambio.index)
    r, cambio = r.loc[idx], cambio.loc[idx]
    filas = []
    for rezago, etiqueta in ((0, "contemporanea"), (1, "rezagada un dia"), (2, "rezagada dos dias")):
        x = cambio.shift(rezago)
        pareja = pd.concat([x, r], axis=1).dropna()
        c = pareja.corr().iloc[0, 1]
        n = len(pareja)
        t = c * np.sqrt(max(n - 2, 1)) / np.sqrt(max(1 - c**2, 1e-12))
        filas.append({"serie": nombre, "alineamiento": etiqueta, "corr": c, "t": t, "n": n})
    return filas

tabla_corr = pd.DataFrame(
    correlaciones(gas, hdd, "MIBGAS vs cambio de HDD")
    + correlaciones(pool, cdd, "OMIE vs cambio de CDD")
)
print(tabla_corr.round(4).to_string(index=False))

                  serie      alineamiento    corr       t    n
MIBGAS vs cambio de HDD     contemporanea  0.0528  2.9157 3045
MIBGAS vs cambio de HDD   rezagada un dia  0.0278  1.5339 3044
MIBGAS vs cambio de HDD rezagada dos dias  0.0506  2.7939 3043
  OMIE vs cambio de CDD     contemporanea  0.0471  2.6161 3074
  OMIE vs cambio de CDD   rezagada un dia  0.0085  0.4734 3073
  OMIE vs cambio de CDD rezagada dos dias -0.0198 -1.0982 3072


La primera mitad de la tesis **se confirma**: la correlación contemporánea
existe y es la mayor de las tres. La segunda mitad —que los rezagos no
predicen— también, en el sentido que importa: las correlaciones rezagadas son
de una magnitud con la que no se construye nada.

Dos advertencias que el escéptico de esta medición dejó claras, y que son más
instructivas que la tabla:

- **La comparación con el simulador estaba mal hecha.** Enfrentaba una cifra
  real restringida a temporada de calefacción contra una cifra simulada de
  muestra completa. Medido en la misma temporada, el simulador infla el canal
  térmico por un factor de entre 2,8 y 3,5.
- **La estrategia sobre el rezago es un billete de lotería.** Con la regla
  `signo(cambio de HDD de ayer)` el Sharpe sale positivo, pero la tasa de
  acierto es del 24,8 %, la mediana del resultado diario es exactamente cero, y
  **el 1 % de los días aporta el 70 % del beneficio**. Además la posición es
  cero la mitad del año, porque en Madrid los HDD valen cero 1 517 de los 3 078
  días. Un Sharpe así no se puede presentar sin esas tres cifras al lado.

## 6. Del error al dinero

El segundo resultado del cuaderno era que ordenar modelos por error y por
dinero da órdenes distintos, y su justificación es algebraica:

$$ \mathrm{MSE}(\hat{y}) = \mathrm{MSE}(0) - 2\,\mathrm{cov}(\hat{y}, y) + \mathrm{var}(\hat{y}) $$

Minimizar el error premia la prudencia en la magnitud; una posición no cobra
por la magnitud, cobra por el signo. Esto no depende de los datos, y se puede
comprobar en una línea.

In [17]:
resultado = validacion_walk_forward(
    crear_modelo("ridge"), X_cal.join(rezagos, how="inner").loc[indice],
    objetivo.loc[indice], particion,
)
pred = resultado.predicciones.dropna()
y_pred = objetivo.loc[pred.index]

print(f"{'factor':>8}  {'MASE':>8}  {'Sharpe neto':>12}")
for factor in (0.1, 1.0, 2.0, 10.0):
    escalada = pred * factor
    mase = error_escalado_medio(y_pred, escalada)
    sharpe = resumen_estrategia(ejecutar_estrategia(escalada, y_pred, coste_bps=2.0))["ratio_sharpe"]
    print(f"{factor:8.1f}  {mase:8.3f}  {sharpe:+12.4f}")

  factor      MASE   Sharpe neto
     0.1     0.673       +0.9909


     1.0     0.649       +0.9909
     2.0     0.810       +0.9909
    10.0     3.504       +0.9909


Multiplicar la predicción por diez arruina el MASE y **deja el Sharpe
exactamente igual**. El mecanismo es real.

Pero la ilustración del cuaderno anterior —que el peor por error era el mejor
por dinero— **no sobrevive**. Con datos reales el signo de la correlación de
rangos depende del mercado: en OMIE los dos órdenes casi coinciden (Spearman
+0,80 con los cuatro modelos del cuaderno, +0,66 con 25 especificaciones), y en
MIBGAS se invierten (−0,80 y −0,26). De diez mediciones, seis salen positivas y
cuatro negativas, con media +0,02. **No hay inversión universal**; hay una
advertencia sobre no elegir modelo por el RMSE, que es distinto.

Y la conclusión que de verdad importa para quien fuese a usar esto:

In [18]:
# Cuanto aniade el aprendizaje automatico sobre una tabla de medias
# por dia de la semana, que es lo que cualquiera haria a mano.
print("OMIE, Sharpe con coste de 2 pb, medido en la seccion 3:")
print(f"  ridge con calendario + rezagos : {filas[2]['Sharpe']:+.3f}")
print(f"  ridge con solo el calendario   : {filas[0]['Sharpe']:+.3f}")
print(f"  regla de cero parametros       : {regla['ratio_sharpe']:+.3f}")
print()
print("La regla sin un solo parametro estimado bate a las dos configuraciones")
print("del modelo. Cuando eso pasa, el resultado no es el modelo: es el patron")
print("que el modelo esta reaprendiendo peor de lo que ya se sabia.")

OMIE, Sharpe con coste de 2 pb, medido en la seccion 3:
  ridge con calendario + rezagos : +0.991
  ridge con solo el calendario   : +0.155
  regla de cero parametros       : +4.412

La regla sin un solo parametro estimado bate a las dos configuraciones
del modelo. Cuando eso pasa, el resultado no es el modelo: es el patron
que el modelo esta reaprendiendo peor de lo que ya se sabia.


## 7. El riesgo: los modelos de la librería suspenden

El cuaderno 02 validó los modelos de VaR con datos simulados y pasaban. Con el
pool español no.

In [19]:
from icm.riesgo.backtesting import backtest_var

informes = []
for metodo in ("historico", "parametrico_normal", "parametrico_t", "cornish_fisher", "ewma"):
    try:
        informes.append(backtest_var(rend, nivel=0.99, ventana=250, metodo=metodo).resumen)
    except Exception as error:
        print(f"  {metodo}: {type(error).__name__}: {str(error)[:70]}")

print("Backtest de VaR al 99 %, ventana movil de 250 sesiones, pool iberico:")
print()
print(pd.DataFrame(informes)[
    ["metodo", "excepciones", "excepciones_esperadas", "p_kupiec", "zona_basilea", "modelo_valido"]
].to_string(index=False))

Backtest de VaR al 99 %, ventana movil de 250 sesiones, pool iberico:

            metodo  excepciones  excepciones_esperadas     p_kupiec zona_basilea  modelo_valido
         historico         55.0                  29.34 2.175301e-05         roja          False
parametrico_normal         75.0                  29.34 1.401324e-12         roja          False
     parametrico_t         35.0                  29.34 3.081062e-01        verde          False
    cornish_fisher         35.0                  29.34 3.081062e-01        verde           True
              ewma         74.0                  29.34 3.681944e-12         roja          False


Y la razón de fondo está en los momentos de la serie.

In [20]:
from icm.cartera.metricas import contraste_jarque_bera
from icm.riesgo.var import dominio_valido_cornish_fisher
from scipy import stats

print(f"Volatilidad anualizada del pool : {rend.std(ddof=1) * np.sqrt(252):.3f}")
print(f"  el CATALOGO del simulador declaraba 0,75 para OMIE")
print()
print(f"Asimetria  {rend.skew():+.3f}")
print(f"Curtosis   {rend.kurtosis():+.3f}  (en exceso sobre la normal)")
jb = contraste_jarque_bera(rend)
print(f"Jarque-Bera: estadistico {jb['estadistico']:.1f}, p = {jb['p_valor']:.2e}")
print()
print(f"Dias con |z| > 5 : {100 * (rend.abs() > 5 * rend.std()).mean():.3f} %")
print(f"  lo que la normal predice: {100 * 2 * stats.norm.sf(5):.7f} %")
print()
gl = stats.t.fit(rend.to_numpy())[0]
print(f"Grados de libertad de la t ajustada: {gl:.2f}")
print("  por debajo de 2 la varianza no es finita: ninguna medida que la use significa nada")
print()
print("Cornish-Fisher dentro de su dominio de validez:",
      dominio_valido_cornish_fisher(rend.skew(), rend.kurtosis()))
print("  ...y es el unico metodo al que el backtest da el visto bueno")

Volatilidad anualizada del pool : 6.385
  el CATALOGO del simulador declaraba 0,75 para OMIE

Asimetria  +0.374
Curtosis   +14.953  (en exceso sobre la normal)
Jarque-Bera: estadistico 29635.1, p = 0.00e+00

Dias con |z| > 5 : 0.628 %
  lo que la normal predice: 0.0000573 %

Grados de libertad de la t ajustada: 1.26
  por debajo de 2 la varianza no es finita: ninguna medida que la use significa nada

Cornish-Fisher dentro de su dominio de validez: False
  ...y es el unico metodo al que el backtest da el visto bueno


Tres de los cinco métodos caen en **zona roja** de Basilea, con 55, 75 y 74
excepciones donde se esperaban 29. El histórico y el normal fallan por lo
previsible: las colas de esta serie no se parecen a nada para lo que se
diseñaron.

Pero fijaos en la única fila con `modelo_valido = True`. Es la de
Cornish-Fisher, y la celda siguiente dice que **su expansión está fuera de su
dominio de validez**. Es un aprobado espurio: el método pasa el contraste
porque su cuantil, mal calculado, resulta ser conservador. Si la librería no
tuviese la comprobación de dominio —que se añadió en la primera revisión—
este sería el modelo que uno elegiría.

## 8. Qué llevarse de aquí

**1. El ratio de Sharpe es la estadística equivocada para este mercado.** La
regla semanal da Sharpe +4,4 en el pool con una rentabilidad compuesta del
−100 % y una caída que desborda el coma flotante, y Sharpe +2,6 en el gas con
+313 % anual y una caída del 57 %. El Sharpe es **mejor** donde el resultado es
la ruina. Con una volatilidad diaria del 640 % anualizado, un cociente de media
sobre desviación típica no ve nada de lo que importa. Las dos columnas de al
lado —`rentabilidad_anualizada` y `maxima_caida`— sí, y el cuaderno anterior no
las informaba.

**2. El resultado más grande del mercado no es cobrable.** El ciclo semanal es
enorme (19 €/MWh de recorrido, t de 13) y públicamente conocido. Bien
dimensionado da +64 % anual al 10 % de volatilidad. Y suma cero a lo largo de la
semana: mide la amplitud de una forma. Para cobrarla habría que almacenar
electricidad, y el contraste con el gas —donde la misma regla **sí** produce
capital compuesto— es la prueba. El instrumento que paga esa señal es una
batería, no un modelo.

**3. Ni la validación honesta ni la ausencia de fugas garantizan que el número
signifique algo.** Todo lo de la sección 3 está medido con walk-forward,
embargo, coste de transacción y volatilidad estimada solo con el pasado. Está
bien hecho. Y sigue sin ser dinero, por una razón que ninguna comprobación
estadística podía detectar: la naturaleza del subyacente. La pregunta «¿se
puede tener esta posición?» no es un detalle de implementación.

**4. Cuando el modelo no bate tres signos puestos a mano, el modelo no es el
resultado.** La regla de cero parámetros da un Sharpe 28 veces mayor que el
ridge con las mismas variables de calendario. Merece la pena medir siempre esa
línea base trivial: es más informativa que cualquier comparación entre
modelos.

**5. El conocimiento del mercado no es opcional.** El mecanismo de ajuste
ibérico afecta al 17,9 % de la muestra y cambia el signo del spark spread
medio: −25,85 €/MWh dentro de esa ventana y +3,37 fuera. No hay validación
cruzada que sustituya a saber que existió.

**6. Los modelos de riesgo suspenden, y el que aprueba lo hace por
equivocación.** Con una curtosis en exceso de 15, un 0,63 % de días más allá de
cinco desviaciones típicas (la normal predice un 0,00006 %) y una t ajustada
con **1,26 grados de libertad** —varianza infinita—, tres de los cinco métodos
caen en zona roja. El único que aprueba es Cornish-Fisher, y su propia
comprobación de dominio dice que la expansión no es válida con estas colas.
Sin esa comprobación, sería el modelo elegido.

**7. Lo que no se puede medir hay que dejarlo sin medir.** La tesis de las
previsiones publicadas era el resultado estrella del cuaderno anterior y aquí
aparece como «no contrastable», porque las series necesarias exigen un token
que no tenemos. Eso es un resultado honesto, y es preferible a sustituirlas por
un sucedáneo que se parezca.

---

*Los datos están en `datos/reales/`, con su procedencia y un SHA-256 por
fichero en `manifiesto.json`. La exploración la hicieron cinco análisis
independientes, cada uno verificado por un escéptico que recalculó las cifras
por su cuenta; los cinco veredictos fueron «confirmado con matices» y varios de
esos matices están en el texto de arriba. Todas las cifras de este cuaderno,
en cambio, se recalcularon una vez más con las funciones de la propia librería
antes de escribirlas: los Sharpe que informaban los análisis se habían
calculado con una anualización propia y no se reproducían con
`resumen_estrategia`, que es la que sale en las celdas.*